# Notebook 01: Data Collection and Setup

---

## Overview

This notebook handles the initial setup and data collection for the NIH Chest X-Ray Disease Detection project.

**Objectives:**
1. Set up project environment and verify installations
2. Configure Kaggle API for dataset download
3. Download NIH Chest X-Ray dataset from Kaggle (112,120 images, ~42 GB)
4. **Download expert-validated labels from Google Cloud Healthcare**
5. Perform initial data inspection
6. Set up directory structure for processed data

**Outputs:**
- Raw data in `data/raw/`
- Expert labels in `data/raw/expert_labels/`
- Initial data quality report
- Directory structure for downstream processing

**Data Sources:**
- **Primary**: NIH Chest X-Ray Dataset (Kaggle) - Original images and text-mined labels
- **Enhanced**: Google Cloud Healthcare Expert Labels - Manually validated annotations by radiologists

---

## 1. Environment Setup and Import Libraries

In [1]:
# Standard library imports
import json
import os
import shutil
import stat
import subprocess
import sys
import warnings
from pathlib import Path

# Third-party downloads
import kagglehub
import requests

# Data manipulation
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Set up plotting style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
warnings.filterwarnings('ignore')

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

print("Libraries imported successfully!")
print(f"Python version: {sys.version}")
print(f"Pandas version: {pd.__version__}")
print(f"NumPy version: {np.__version__}")

Libraries imported successfully!
Python version: 3.12.11 (main, Sep 19 2025, 02:30:54) [Clang 17.0.0 (clang-1700.3.19.1)]
Pandas version: 2.3.3
NumPy version: 2.0.2


## 2. Set Up Project Paths

In [2]:
# Define base paths
PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / 'data'
RAW_DATA_DIR = DATA_DIR / 'raw'
PROCESSED_DATA_DIR = DATA_DIR / 'processed'
MODELS_DIR = PROJECT_ROOT / 'models'
OUTPUTS_DIR = PROJECT_ROOT / 'outputs'

# Create directories if they don't exist
directories = [
    RAW_DATA_DIR,
    PROCESSED_DATA_DIR,
    PROCESSED_DATA_DIR / 'train',
    PROCESSED_DATA_DIR / 'val',
    PROCESSED_DATA_DIR / 'test',
    DATA_DIR / 'sample',
    MODELS_DIR / 'saved_models',
    MODELS_DIR / 'training_history',
    OUTPUTS_DIR / 'figures',
    OUTPUTS_DIR / 'reports',
    OUTPUTS_DIR / 'predictions'
]

for directory in directories:
    directory.mkdir(parents=True, exist_ok=True)
    print(f"✓ Created/verified: {directory}")

print("\n📁 Project structure set up successfully!")

✓ Created/verified: /Users/james/CodeInstitute/data/raw
✓ Created/verified: /Users/james/CodeInstitute/data/processed
✓ Created/verified: /Users/james/CodeInstitute/data/processed/train
✓ Created/verified: /Users/james/CodeInstitute/data/processed/val
✓ Created/verified: /Users/james/CodeInstitute/data/processed/test
✓ Created/verified: /Users/james/CodeInstitute/data/sample
✓ Created/verified: /Users/james/CodeInstitute/models/saved_models
✓ Created/verified: /Users/james/CodeInstitute/models/training_history
✓ Created/verified: /Users/james/CodeInstitute/outputs/figures
✓ Created/verified: /Users/james/CodeInstitute/outputs/reports
✓ Created/verified: /Users/james/CodeInstitute/outputs/predictions

📁 Project structure set up successfully!


## 3. Configure Kaggle API

⚠️ **Important:** Kaggle API credentials are stored in the **project directory** at `.kaggle/kaggle.json`

This allows each project to have its own credentials and makes setup easier for assessors.

In [3]:
# Check for Kaggle API credentials in project directory
kaggle_config_path = PROJECT_ROOT / '.kaggle' / 'kaggle.json'

if kaggle_config_path.exists():
    print(f"✓ Kaggle API credentials found at: {kaggle_config_path}")
    
    # Verify permissions (should be 600)
    current_permissions = oct(os.stat(kaggle_config_path).st_mode)[-3:]
    print(f"  Current permissions: {current_permissions}")
    
    if current_permissions != '600':
        os.chmod(kaggle_config_path, stat.S_IRUSR | stat.S_IWUSR)
        print("  ✓ Fixed permissions to 600")
    
    # ⚡ CRITICAL: Set environment variable BEFORE any kaggle imports
    # This must be done before the kaggle module is loaded
    os.environ['KAGGLE_CONFIG_DIR'] = str(kaggle_config_path.parent)
    print(f"\n✓ KAGGLE_CONFIG_DIR set to: {kaggle_config_path.parent}")
    print("  This will be used for Kaggle API authentication in the next cell")
    
else:
    print("❌ Kaggle API credentials not found!")
    print(f"   Expected location: {kaggle_config_path}")
    print()
    print("Setup instructions:")
    print("1. Create directory:")
    print(f"   mkdir -p {kaggle_config_path.parent}")
    print()
    print("2. Copy your kaggle.json to project:")
    print(f"   cp ~/.kaggle/kaggle.json {kaggle_config_path}")
    print()
    print("   OR download from: https://www.kaggle.com/settings/account")
    print(f"   and save to: {kaggle_config_path}")
    print()
    print("3. Set permissions:")
    print(f"   chmod 600 {kaggle_config_path}")

❌ Kaggle API credentials not found!
   Expected location: /Users/james/CodeInstitute/.kaggle/kaggle.json

Setup instructions:
1. Create directory:
   mkdir -p /Users/james/CodeInstitute/.kaggle

2. Copy your kaggle.json to project:
   cp ~/.kaggle/kaggle.json /Users/james/CodeInstitute/.kaggle/kaggle.json

   OR download from: https://www.kaggle.com/settings/account
   and save to: /Users/james/CodeInstitute/.kaggle/kaggle.json

3. Set permissions:
   chmod 600 /Users/james/CodeInstitute/.kaggle/kaggle.json


In [4]:
# Import and authenticate Kaggle API
# The KAGGLE_CONFIG_DIR environment variable was set in the previous cell

try:
    from kaggle.api.kaggle_api_extended import KaggleApi
    
    api = KaggleApi()
    api.authenticate()
    
    print("✅ Kaggle API authenticated successfully!")
    print(f"   Using credentials from: {kaggle_config_path}")
    
    # Verify the API is actually using our config
    print("\n📁 Active Kaggle config directory:")
    print(f"   {os.environ.get('KAGGLE_CONFIG_DIR', 'Not set')}")
    
except Exception as e:
    print("❌ Kaggle API authentication failed!")
    print(f"\nError message: {e}")
    print()
    print("🔍 Diagnostics:")
    print(f"  KAGGLE_CONFIG_DIR: {os.environ.get('KAGGLE_CONFIG_DIR', 'NOT SET')}")
    print(f"  Expected file: {kaggle_config_path}")
    print(f"  File exists: {kaggle_config_path.exists()}")
    
    if kaggle_config_path.exists():
        try:
            with open(kaggle_config_path) as f:
                config = json.load(f)
                print("  Config valid JSON: ✓")
                print(f"  Has 'username' key: {'username' in config}")
                print(f"  Has 'key' key: {'key' in config}")
        except Exception as json_error:
            print(f"  Config valid JSON: ✗ ({json_error})")
    
    print()
    print("📝 Troubleshooting:")
    print("1. Verify file permissions (should be 600)")
    print("2. Check JSON format (needs 'username' and 'key' fields)")
    print("3. Accept dataset terms: https://www.kaggle.com/datasets/nih-chest-xrays/data")
    print()
    print("⚠️  Note: Kaggle API has a known limitation - it reads config location")
    print("   when the module is first imported. If this persists:")
    print("   - Restart the Jupyter kernel (Kernel > Restart)")
    print("   - OR use the Kaggle CLI directly instead:")
    print(f"     kaggle datasets download -d nih-chest-xrays/data -p {RAW_DATA_DIR}")

❌ Kaggle API authentication failed!

Error message: Could not find kaggle.json. Make sure it's located in /Users/james/.kaggle. Or use the environment method. See setup instructions at https://github.com/Kaggle/kaggle-api/

🔍 Diagnostics:
  KAGGLE_CONFIG_DIR: NOT SET
  Expected file: /Users/james/CodeInstitute/.kaggle/kaggle.json
  File exists: False

📝 Troubleshooting:
1. Verify file permissions (should be 600)
2. Check JSON format (needs 'username' and 'key' fields)
3. Accept dataset terms: https://www.kaggle.com/datasets/nih-chest-xrays/data

⚠️  Note: Kaggle API has a known limitation - it reads config location
   when the module is first imported. If this persists:
   - Restart the Jupyter kernel (Kernel > Restart)
   - OR use the Kaggle CLI directly instead:
     kaggle datasets download -d nih-chest-xrays/data -p /Users/james/CodeInstitute/data/raw


## 4. Download NIH Chest X-Ray Dataset

**Dataset:** `nih-chest-xrays/data`
- **Size:** ~42 GB (compressed), 112,120 images
- **Images:** 1024x1024 PNG grayscale
- **Labels:** 15 classes (14 diseases + "No Finding")

**Note:** This download may take 30-60 minutes depending on your internet connection.

In [5]:
# Dataset information
DATASET_NAME = 'nih-chest-xrays/data'
DOWNLOAD_PATH = str(RAW_DATA_DIR)

print(f"Dataset: {DATASET_NAME}")
print(f"Download path: {DOWNLOAD_PATH}")
print("\nThis will download approximately 42 GB of data.")
print("Download time estimate: 30-60 minutes (depending on connection speed)")

Dataset: nih-chest-xrays/data
Download path: /Users/james/CodeInstitute/data/raw

This will download approximately 42 GB of data.
Download time estimate: 30-60 minutes (depending on connection speed)


In [6]:
# Download dataset using kagglehub (modern Kaggle library)
metadata_file = RAW_DATA_DIR / 'Data_Entry_2017.csv'

if metadata_file.exists():
    print("✓ Dataset already downloaded!")
    print(f"  Metadata file found: {metadata_file}")
else:
    print("⏳ Downloading NIH Chest X-Ray dataset...")
    print("   Dataset: nih-chest-xrays/data (~42 GB)")
    print("   This will take 30-60 minutes depending on connection speed")
    print("   📡 Using Kaggle's modern download library with built-in progress\n")
    
    try:
        # Download dataset - kagglehub handles progress display automatically
        # It downloads to a cache location and returns the path
        print("🔐 Authenticating with Kaggle...")
        download_path = kagglehub.dataset_download("nih-chest-xrays/data")
        
        print("\n✓ Download complete!")
        print(f"  Dataset cached at: {download_path}")
        print()
        
        # Copy/move files to our project structure
        print("📂 Organizing files into project structure...")
        source_path = Path(download_path)
        
        # Find all files and copy to RAW_DATA_DIR
        files_copied = 0
        for source_file in source_path.rglob('*'):
            if source_file.is_file():
                # Preserve directory structure
                relative_path = source_file.relative_to(source_path)
                dest_file = RAW_DATA_DIR / relative_path
                dest_file.parent.mkdir(parents=True, exist_ok=True)
                
                if not dest_file.exists():
                    shutil.copy2(source_file, dest_file)
                    files_copied += 1
        
        print(f"✓ Copied {files_copied:,} files to {RAW_DATA_DIR}")
        
        # Verify metadata file exists
        if metadata_file.exists():
            final_size = sum(
                f.stat().st_size 
                for f in RAW_DATA_DIR.rglob('*') 
                if f.is_file()
            ) / (1024**3)  # GB
            
            print(f"\n{'='*60}")
            print("✅ DOWNLOAD SUCCESSFUL!")
            print(f"{'='*60}")
            print(f"✓ Total size: {final_size:.2f} GB")
            print(f"✓ Metadata file: {metadata_file}")
            print("✓ Ready for analysis!\n")
        else:
            print(f"\n⚠️ Warning: Metadata file not found at {metadata_file}")
            print(f"   Files were downloaded to: {download_path}")
            print("   Please check the downloaded files manually")
            
    except Exception as e:
        print(f"\n❌ Download failed: {e}")
        print()
        print("Troubleshooting:")
        print("1. Ensure kagglehub is installed: pip install kagglehub")
        print("2. Check Kaggle credentials are set up correctly")
        print("3. Accept dataset terms: https://www.kaggle.com/datasets/nih-chest-xrays/data")
        print("4. Check internet connection")
        print()
        print("Note: kagglehub uses the same credentials as the kaggle CLI")
        print(f"      Credentials location: {kaggle_config_path}")
        raise

print("\n✓ Ready for data exploration!")
print("  Next: Run notebook 02_exploratory_data_analysis.ipynb")

✓ Dataset already downloaded!
  Metadata file found: /Users/james/CodeInstitute/data/raw/Data_Entry_2017.csv

✓ Ready for data exploration!
  Next: Run notebook 02_exploratory_data_analysis.ipynb


## 4b. Download Expert Labels from Google Cloud

**Important**: The original NIH labels were automatically extracted from radiological reports using NLP, which introduces labeling noise (~90% accuracy). Google Cloud Healthcare has provided **expert-validated labels** that are manually annotated by radiologists through rigorous adjudication processes.

**Source**: [Google Cloud Healthcare - NIH Chest X-Ray Additional Labels](https://cloud.google.com/healthcare-api/docs/resources/public-datasets/nih-chest#additional_labels)

**What we're downloading**:

1. **Four Findings Expert Labels** (4,374 images)
   - **Publication**: Majkowska et al., *Radiology*, 2019
   - **Findings**: Airspace opacity, pneumothorax, nodule/mass, fracture
   - **Process**: Adjudicated review by 3 radiologists (cohort of 11+ board-certified)
   - **Paper**: [doi:10.1148/radiol.2019191293](https://pubs.rsna.org/doi/10.1148/radiol.2019191293)

2. **All Findings Expert Labels** (810 images)
   - **Publication**: Nabulsi et al., *Nature Scientific Reports*, 2021
   - **Findings**: All 14 pathologies + normal/abnormal
   - **Process**: 5 board-certified radiologists, majority vote
   - **Paper**: [doi:10.1038/s41598-021-93967-2](https://arxiv.org/abs/2010.11375)

**Why this matters**: Expert labels provide higher-quality ground truth for model training and validation, reducing false positives/negatives in critical diagnoses. These labels are used as benchmarks in peer-reviewed publications.

In [7]:
# Download expert labels from Google Cloud Storage
# Create directory for expert labels
expert_labels_dir = RAW_DATA_DIR / 'expert_labels'
expert_labels_dir.mkdir(exist_ok=True)

# Check if expert labels are already downloaded
four_findings_dir = expert_labels_dir / 'four_findings_expert_labels'
all_findings_dir = expert_labels_dir / 'all_findings_expert_labels'
readme_file = expert_labels_dir / 'README.md'

# Check if all expected components exist
labels_already_downloaded = (
    four_findings_dir.exists() and 
    all_findings_dir.exists() and
    len(list(four_findings_dir.glob('*.csv'))) > 0 and
    len(list(all_findings_dir.glob('*.csv'))) > 0
)

if labels_already_downloaded:
    print("✓ Expert labels already downloaded!")
    print(f"  Location: {expert_labels_dir}\n")
    
    # Show what's available
    print("📂 Available expert label sets:")
    for item in sorted(expert_labels_dir.iterdir()):
        if item.is_file():
            size_kb = item.stat().st_size / 1024
            print(f"  📄 {item.name:<40} ({size_kb:>8.2f} KB)")
        elif item.is_dir():
            csv_files = list(item.glob('*.csv'))
            num_files = len(list(item.rglob('*')))
            print(f"  📁 {item.name:<40} ({len(csv_files)} CSV files, {num_files} total)")
    
    print("\n💡 Tip: Expert labels will be loaded and compared with NIH labels in Notebook 02")

else:
    print("📥 Downloading expert labels from Google Cloud Storage...")
    print(f"   Destination: {expert_labels_dir}\n")
    
    # GCS bucket details
    GCS_BUCKET = "gcs-public-data--healthcare-nih-chest-xray-labels"
    GCS_BASE_URL = f"https://storage.googleapis.com/{GCS_BUCKET}"
    
    # Files/directories to download
    items_to_download = [
        "README.md",
        "four_findings_expert_labels",
        "all_findings_expert_labels"
    ]
    
    # Check if gsutil is available
    def check_gsutil():
        try:
            subprocess.run(['gsutil', 'version'], capture_output=True, check=True)
            return True
        except (subprocess.CalledProcessError, FileNotFoundError):
            return False
    
    has_gsutil = check_gsutil()
    
    if has_gsutil:
        print("✓ gsutil detected - using Google Cloud Storage CLI")
        print("  (Faster and more reliable for large files)\n")
        
        try:
            # Use gsutil to download
            for item in items_to_download:
                gcs_path = f"gs://{GCS_BUCKET}/{item}"
                print(f"Downloading: {item}")
                
                # Check if it's a directory or file
                if item.endswith('.md'):
                    # Single file
                    result = subprocess.run(
                        ['gsutil', 'cp', gcs_path, str(expert_labels_dir / item)],
                        capture_output=True,
                        text=True
                    )
                else:
                    # Directory - use recursive copy
                    result = subprocess.run(
                        ['gsutil', '-m', 'cp', '-r', gcs_path, str(expert_labels_dir)],
                        capture_output=True,
                        text=True
                    )
                
                if result.returncode == 0:
                    print(f"  ✓ Downloaded: {item}")
                else:
                    print(f"  ⚠️  Warning: {item} - {result.stderr}")
            
            print(f"\n✅ Expert labels downloaded to: {expert_labels_dir}")
            
        except Exception as e:
            print(f"\n❌ gsutil download failed: {e}")
            print("Falling back to HTTP download...")
            has_gsutil = False
    
    if not has_gsutil:
        print("ℹ️  gsutil not available - using HTTP download")
        print("  (Note: This is slower. Install gsutil for better performance)")
        print("  Install: pip install gsutil\n")
        
        # Download README.md via HTTP
        try:
            readme_url = f"{GCS_BASE_URL}/README.md"
            print("Downloading: README.md")
            response = requests.get(readme_url, timeout=30)
            response.raise_for_status()
            
            with open(expert_labels_dir / 'README.md', 'wb') as f:
                f.write(response.content)
            print("  ✓ Downloaded: README.md\n")
            
        except Exception as e:
            print(f"  ⚠️  Could not download README.md: {e}\n")
        
        # For directories, we need to list files first
        print("⚠️  Note: HTTP download of directory contents requires listing files")
        print("   For complete expert labels, please:")
        print("   1. Install gsutil: pip install gsutil")
        print("   2. Re-run this cell")
        print()
        print("   OR download manually from:")
        print(f"   https://console.cloud.google.com/storage/browser/{GCS_BUCKET}")
        print()
        print("   OR use gsutil directly:")
        print(f"   gsutil -m cp -r gs://{GCS_BUCKET}/* {expert_labels_dir}")
    
    # Verify what was downloaded
    print("\n📂 Contents of expert_labels directory:")
    downloaded_items = list(expert_labels_dir.iterdir())
    if downloaded_items:
        for item in sorted(downloaded_items):
            if item.is_file():
                size_kb = item.stat().st_size / 1024
                print(f"  📄 {item.name:<40} ({size_kb:>8.2f} KB)")
            elif item.is_dir():
                csv_files = list(item.glob('*.csv'))
                num_files = len(list(item.rglob('*')))
                print(f"  📁 {item.name:<40} ({len(csv_files)} CSV files, {num_files} total)")
    else:
        print("  (No files downloaded yet)")
    
    print("\n💡 Tip: Expert labels will be loaded and compared with NIH labels in Notebook 02")

✓ Expert labels already downloaded!
  Location: /Users/james/CodeInstitute/data/raw/expert_labels

📂 Available expert label sets:
  📄 README.md                                (    1.37 KB)
  📁 all_findings_expert_labels               (2 CSV files, 4 total)
  📁 four_findings_expert_labels              (3 CSV files, 10 total)

💡 Tip: Expert labels will be loaded and compared with NIH labels in Notebook 02


## 5. Initial Data Inspection

In [8]:
# List files in raw data directory with enhanced stats
print("📂 Files in raw data directory:\n")
for item in sorted(RAW_DATA_DIR.iterdir()):
    if item.is_file():
        size_mb = item.stat().st_size / (1024 * 1024)
        print(f"  📄 {item.name:<35} {size_mb:>8.2f} MB")
    elif item.is_dir():
        # Count ALL files recursively (not just immediate children)
        all_files = [f for f in item.rglob('*') if f.is_file()]
        num_files = len(all_files)
        
        # Calculate total directory size
        total_size_mb = sum(f.stat().st_size for f in all_files) / (1024 * 1024)
        
        print(f"  📁 {item.name:<35} {num_files:>6,} files  ({total_size_mb:>8.2f} MB)")

# Summary statistics
all_files = [f for f in RAW_DATA_DIR.rglob('*') if f.is_file()]
total_files = len(all_files)
total_size_gb = sum(f.stat().st_size for f in all_files) / (1024**3)

print(f"\n{'─'*60}")
print(f"  📊 Total: {total_files:,} files | {total_size_gb:.2f} GB")
print(f"{'─'*60}")

📂 Files in raw data directory:

  📄 ARXIV_V5_CHESTXRAY.pdf                  8.55 MB
  📄 BBox_List_2017.csv                      0.09 MB
  📄 Data_Entry_2017.csv                     7.50 MB
  📄 FAQ_CHESTXRAY.pdf                       0.07 MB
  📄 LOG_CHESTXRAY.pdf                       0.00 MB
  📄 README_CHESTXRAY.pdf                    0.81 MB
  📁 expert_labels                           14 files  (    1.78 MB)
  📁 images_001                           4,999 files  ( 1914.94 MB)
  📁 images_002                          10,000 files  ( 3768.03 MB)
  📁 images_003                          10,000 files  ( 3746.66 MB)
  📁 images_004                          10,000 files  ( 3659.50 MB)
  📁 images_005                          10,000 files  ( 3751.06 MB)
  📁 images_006                          10,000 files  ( 3799.34 MB)
  📁 images_007                          10,000 files  ( 3828.18 MB)
  📁 images_008                          10,000 files  ( 3829.91 MB)
  📁 images_009                          10,0

### 5.1 Load Metadata

In [9]:
# Load main metadata file
metadata_df = pd.read_csv(RAW_DATA_DIR / 'Data_Entry_2017.csv')

print(f"✓ Loaded metadata: {metadata_df.shape[0]:,} records, {metadata_df.shape[1]} columns")
print(f"\nColumn names:\n{metadata_df.columns.tolist()}")

✓ Loaded metadata: 112,120 records, 12 columns

Column names:
['Image Index', 'Finding Labels', 'Follow-up #', 'Patient ID', 'Patient Age', 'Patient Gender', 'View Position', 'OriginalImage[Width', 'Height]', 'OriginalImagePixelSpacing[x', 'y]', 'Unnamed: 11']


In [10]:
# Display first few rows
print("\n📊 First 5 records:")
metadata_df.head()


📊 First 5 records:


,Image Index,Finding Labels,Follow-up #,Patient ID,Patient Age,Patient Gender,View Position,OriginalImage[Width,Height],OriginalImagePixelSpacing[x,y],Unnamed: 11
0,00000001_000.png,Cardiomegaly,0,1,58,M,PA,2682,2749,0.143,0.143,NaN
1,00000001_001.png,Cardiomegaly|Emphysema,1,1,58,M,PA,2894,2729,0.143,0.143,NaN
2,00000001_002.png,Cardiomegaly|Effusion,2,1,58,M,PA,2500,2048,0.168,0.168,NaN
3,00000002_000.png,No Finding,0,2,81,M,PA,2500,2048,0.171,0.171,NaN
4,00000003_000.png,Hernia,0,3,81,F,PA,2582,2991,0.143,0.143,NaN


In [11]:
# Basic statistics
print("\n📈 Dataset Statistics:")
print(f"  Total images: {len(metadata_df):,}")
print(f"  Unique patients: {metadata_df['Patient ID'].nunique():,}")
print(f"  Age range: {metadata_df['Patient Age'].min():.0f} - {metadata_df['Patient Age'].max():.0f} years")
print("  Gender distribution:")
print(metadata_df['Patient Gender'].value_counts())


📈 Dataset Statistics:
  Total images: 112,120
  Unique patients: 30,805
  Age range: 1 - 414 years
  Gender distribution:
Patient Gender
M    63340
F    48780
Name: count, dtype: int64


### 5.2 Inspect Disease Labels

In [12]:
# Extract unique disease labels
all_labels = metadata_df['Finding Labels'].str.split('|')
unique_diseases = sorted(set([label for labels in all_labels for label in labels]))

print(f"\n🏥 Disease Labels ({len(unique_diseases)} classes):\n")
for i, disease in enumerate(unique_diseases, 1):
    count = sum(metadata_df['Finding Labels'].str.contains(disease, regex=False))
    percentage = (count / len(metadata_df)) * 100
    print(f"  {i:2d}. {disease:<25} {count:>7,} images ({percentage:>5.2f}%)")


🏥 Disease Labels (15 classes):

   1. Atelectasis                11,559 images (10.31%)
   2. Cardiomegaly                2,776 images ( 2.48%)
   3. Consolidation               4,667 images ( 4.16%)
   4. Edema                       2,303 images ( 2.05%)
   5. Effusion                   13,317 images (11.88%)
   6. Emphysema                   2,516 images ( 2.24%)
   7. Fibrosis                    1,686 images ( 1.50%)
   8. Hernia                        227 images ( 0.20%)
   9. Infiltration               19,894 images (17.74%)
  10. Mass                        5,782 images ( 5.16%)
  11. No Finding                 60,361 images (53.84%)
  12. Nodule                      6,331 images ( 5.65%)
  13. Pleural_Thickening          3,385 images ( 3.02%)
  14. Pneumonia                   1,431 images ( 1.28%)
  15. Pneumothorax                5,302 images ( 4.73%)


In [13]:
# Multi-label statistics
label_counts = all_labels.apply(len)

print("\n📊 Multi-Label Statistics:")
print(f"  Images with 1 label:  {(label_counts == 1).sum():,}")
print(f"  Images with 2 labels: {(label_counts == 2).sum():,}")
print(f"  Images with 3 labels: {(label_counts == 3).sum():,}")
print(f"  Images with 4+ labels: {(label_counts >= 4).sum():,}")
print(f"  Max labels per image: {label_counts.max()}")


📊 Multi-Label Statistics:
  Images with 1 label:  91,324
  Images with 2 labels: 14,306
  Images with 3 labels: 4,856
  Images with 4+ labels: 1,634
  Max labels per image: 9


### 5.3 Check Bounding Box Annotations (Optional)

In [14]:
# Load bounding box file if available
bbox_file = RAW_DATA_DIR / 'BBox_List_2017.csv'

if bbox_file.exists():
    bbox_df = pd.read_csv(bbox_file)
    print(f"✓ Bounding box data loaded: {len(bbox_df):,} annotations")
    print("\nFirst few bounding boxes:")
    display(bbox_df.head())
else:
    print("ℹ️ Bounding box file not found (optional for this project)")

✓ Bounding box data loaded: 984 annotations

First few bounding boxes:


,Image Index,Finding Label,Bbox [x,y,w,h],Unnamed: 6,Unnamed: 7,Unnamed: 8
0,00013118_008.png,Atelectasis,225.084746,547.019217,86.779661,79.186441,NaN,NaN,NaN
1,00014716_007.png,Atelectasis,686.101695,131.543498,185.491525,313.491525,NaN,NaN,NaN
2,00029817_009.png,Atelectasis,221.830508,317.053115,155.118644,216.949153,NaN,NaN,NaN
3,00014687_001.png,Atelectasis,726.237288,494.951420,141.016949,55.322034,NaN,NaN,NaN
4,00017877_001.png,Atelectasis,660.067797,569.780787,200.677966,78.101695,NaN,NaN,NaN


## 6. Save Initial Data Report

In [15]:
# Create initial data report
report = {
    'total_images': int(len(metadata_df)),
    'unique_patients': int(metadata_df['Patient ID'].nunique()),
    'age_min': int(metadata_df['Patient Age'].min()),
    'age_max': int(metadata_df['Patient Age'].max()),
    'age_mean': float(metadata_df['Patient Age'].mean()),
    'gender_distribution': {
        str(k): int(v) for k, v in metadata_df['Patient Gender'].value_counts().to_dict().items()
    },
    'num_disease_classes': int(len(unique_diseases)),
    'disease_classes': unique_diseases,
    'multi_label_images': int((label_counts > 1).sum()),
    'max_labels_per_image': int(label_counts.max())
}

# Save report
report_path = OUTPUTS_DIR / 'reports' / '01_initial_data_report.json'
with open(report_path, 'w') as f:
    json.dump(report, f, indent=2)

print(f"✓ Initial data report saved to: {report_path}")

✓ Initial data report saved to: /Users/james/CodeInstitute/outputs/reports/01_initial_data_report.json


## 7. Summary and Next Steps

### Completed ✅
- Environment setup and library imports
- Project directory structure created
- Kaggle API configured
- NIH Chest X-Ray dataset downloaded (112,120 images)
- **Expert labels downloaded from Google Cloud** (higher quality annotations)
- Initial data inspection performed
- Metadata loaded and analyzed
- Data report generated

### Key Findings 📊
- Dataset contains 112,120 chest X-ray images
- 30,805 unique patients
- 15 disease classes (14 pathologies + "No Finding")
- Multi-label classification challenge
- Significant class imbalance ("No Finding" is majority class)
- **Expert labels available for improved model training**

### Important Notes 💡
- **Two label sources**: Original NIH labels (~90% accuracy) + Expert labels (manually validated)
- Expert labels provide ground truth for select findings
- Models trained on expert labels expected to have higher precision/recall
- Next notebook will compare label agreement between sources

### Next Notebook 📝
**02_exploratory_data_analysis.ipynb**
- Detailed statistical analysis of demographics
- Disease distribution visualization
- **Label quality comparison** (NIH vs. Expert labels)
- Co-occurrence analysis
- Sample image visualization
- Data quality assessment

In [16]:
print("="*60)
print("  ✅ Notebook 01 Complete: Data Collection and Setup")
print("="*60)
print(f"\nData location: {RAW_DATA_DIR}")
print(f"Total images: {len(metadata_df):,}")
print("Ready for EDA in Notebook 02!")

  ✅ Notebook 01 Complete: Data Collection and Setup

Data location: /Users/james/CodeInstitute/data/raw
Total images: 112,120
Ready for EDA in Notebook 02!
